In [1]:
import time, pickle
import tracemalloc
import numpy as np
from pathlib import Path
from scipy import sparse
from sklearn.datasets import fetch_openml
from sklearn.neighbors import NearestNeighbors
from flyhash import FlyHash

TRAIN_SIZE, TEST_SIZE, K_NEIGHBORS, SEED = 10_000, 500, 10, 42
HASH_DIM, SPARSITY = 7840, 0.05

X_mnist, _ = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False, parser="auto")
X_mnist = X_mnist.astype(np.float32) / 255.0

perm = np.random.default_rng(SEED).permutation(len(X_mnist))
train_idx = perm[:TRAIN_SIZE]
test_idx = perm[TRAIN_SIZE:TRAIN_SIZE + TEST_SIZE]
X_train, X_test = X_mnist[train_idx], X_mnist[test_idx]
print(X_train.shape, X_test.shape)

(10000, 784) (500, 784)


In [2]:
def profile(func, repeats=3):
    """Время: медиана по repeats прогонов без tracemalloc.
       Память: отдельный прогон под tracemalloc (пик Python/NumPy/SciPy-аллокаций, МБ)."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = func()
        times.append(time.perf_counter() - t0)
    tracemalloc.start()
    func()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, float(np.median(times)), peak / 1024**2

In [3]:
def fit_bf():
    nn = NearestNeighbors(n_neighbors=K_NEIGHBORS, algorithm="brute", metric="euclidean")
    return nn.fit(X_train)

nn_bf, t_fit_bf, mem_fit_bf = profile(fit_bf)
(distances, indices), t_search_bf, mem_search_bf = profile(lambda: nn_bf.kneighbors(X_test))

print(f"Fit:    {t_fit_bf:.4f} с | пик {mem_fit_bf:.2f} МБ")
print(f"Search: {t_search_bf:.4f} с | пик {mem_search_bf:.2f} МБ")

BASE_PATH = Path("./weights"); BASE_PATH.mkdir(exist_ok=True)
full_path = BASE_PATH / "mnist_exact_k10.pkl"
with open(full_path, "wb") as f:
    pickle.dump({
        "indices": indices, "distances": distances, "K": K_NEIGHBORS,
        "train_idx": train_idx, "test_idx": test_idx,
        "time_fit_sec": t_fit_bf, "time_search_sec": t_search_bf,
        "peak_mem_fit_mb": mem_fit_bf, "peak_mem_search_mb": mem_search_bf,
    }, f)
print("Сохранено:", full_path)

Fit:    0.0059 с | пик 0.03 МБ
Search: 0.0924 с | пик 0.16 МБ
Сохранено: weights\mnist_exact_k10.pkl


In [4]:
def load_ground_truth(path):
    with open(path, "rb") as f:
        return pickle.load(f)["indices"]

def recall_at_k(exact, approx, k):
    return float(np.mean([len(set(e[:k]) & set(a[:k])) / k for e, a in zip(exact, approx)]))

In [5]:
exact_indices = load_ground_truth(full_path)

def hash_to_sparse(model, X):
    hs = model(X)
    hs = hs if sparse.issparse(hs) else sparse.csr_matrix(hs)
    hs = (hs > 0).astype(np.float32).tocsr()
    return hs

def train_fly():
    model = FlyHash(input_dim=X_train.shape[1], hash_dim=HASH_DIM, sparsity=SPARSITY, seed=SEED)
    return model, hash_to_sparse(model, X_train)   # создание проекции + хэширование train

(fly, train_hashes), t_train_fly, mem_train_fly = profile(train_fly)

# Проверка корректности WTA: у каждого объекта должно быть ровно 392 активных бита
nnz = np.asarray(train_hashes.sum(axis=1)).ravel()
print("Активных бит (min/max):", nnz.min(), nnz.max(), "| ожидается", int(HASH_DIM * SPARSITY))

def search_fly(k=K_NEIGHBORS):
    q = hash_to_sparse(fly, X_test)
    overlap = q.dot(train_hashes.T).toarray()
    qn = np.asarray(q.sum(axis=1)).ravel()[:, None]
    dn = np.asarray(train_hashes.sum(axis=1)).ravel()[None, :]
    ham = qn + dn - 2 * overlap                      # расстояние Хэмминга
    part = np.argpartition(ham, k - 1, axis=1)[:, :k]
    order = np.argsort(np.take_along_axis(ham, part, axis=1), axis=1, kind="stable")
    return np.take_along_axis(part, order, axis=1)

approx_indices, t_search_fly, mem_search_fly = profile(search_fly)

index_mb = (train_hashes.data.nbytes + train_hashes.indices.nbytes + train_hashes.indptr.nbytes) / 1024**2
print(f"Обучение: {t_train_fly:.4f} с | пик {mem_train_fly:.2f} МБ")
print(f"Поиск:    {t_search_fly:.4f} с | пик {mem_search_fly:.2f} МБ")
print(f"Размер индекса (train-хэши): {index_mb:.2f} МБ")
for k in (1, 5, 10):
    print(f"Recall@{k}: {recall_at_k(exact_indices, approx_indices, k):.4f}")

Активных бит (min/max): 392.0 392.0 | ожидается 392
Обучение: 2.5736 с | пик 604.28 МБ
Поиск:    1.2881 с | пик 77.95 МБ
Размер индекса (train-хэши): 29.95 МБ
Recall@1: 0.5640
Recall@5: 0.6388
Recall@10: 0.6500
